# Chain-ladder developed submissions
This notebook re-runs the two best `home_insurance_model` approaches on the **developed** claims target from `data/claims.duckdb` (built in `workings/chainladder`, Benktander cost factors by line and loss month):

| New file | Same approach as | What it is |
| --- | --- | --- |
| `chainladder_submission_v001.csv` | `home_insurance_submission_v003` | Final Tweedie GLM (same factors), each line rescaled to the v002 total. Only the risk ranking can change. |
| `chainladder_submission_v002.csv` | `home_insurance_submission_v005` | GLM/XGBoost blend (weight picked on the 2024 holdout) × large-loss loading × trend to 2025. |

The only change from the original approach is the target. `cost_capped` (reported net incurred, capped at the 99.5th percentile) becomes `cost_dev_capped`: developed net incurred (`net_incurred_dev`), capped at the 99.5th percentile of developed claim cost. Everything else is kept the same: training 2021-23, holdout 2024, GLM factor list, XGBoost parameters, blend grid and trend rule. The development factors only scale up claims that have already been reported. So late-reported (IBNR) claims add to severity on rows that already have claims, not to frequency.

In [0]:
%pip install -q xgboost duckdb==1.5.6
dbutils.library.restartPython()

## Step 1: Modelling frames with the developed target
The build is the same as `home_insurance_model` Step 1: exposure standardised to days / 365, then joins to addresses, customers and weather, and the same feature list. The difference is where the claims come from. They are read from the `claims` table in `data/claims.duckdb`. That table is already matched to `row_id` (`policy_id` + `loss_date` within the period) and holds `net_incurred_dev`. Repudiated claims are excluded.

Columns on each training row:
* `cost_reported`: reported net incurred, the same as `cost` in the original notebook.
* `cost_dev`: developed net incurred. Its total, divided by the `cost_reported` total, is the development uplift.
* `cost_dev_capped`: the model target. Each developed claim is capped at the 99.5th percentile for its line.

In [0]:
import os, gc, time, shutil, tempfile
import numpy as np
import pandas as pd
import duckdb

LINES = ['buildings', 'contents', 'motor']
policy_type_map = {'B': 'buildings', 'C': 'contents', 'M': 'motor'}
DATA, OUT_DIR = '../data', '../submissions'
CAP_Q = 0.995
TARGET = 'cost_dev_capped'

# --- Claims (developed) from the shared DuckDB file; read from a local copy ---
tmp_db = os.path.join(tempfile.mkdtemp(), 'claims.duckdb')
shutil.copyfile(f'{DATA}/claims.duckdb', tmp_db)
with duckdb.connect(tmp_db, read_only=True) as con:
    clm = con.sql("""SELECT claim_id, policy_id, line, row_id, loss_date, status, net_incurred, cost_dev_factor, net_incurred_dev
                     FROM claims WHERE status <> 'repudiated'""").df()
caps_dev = clm.groupby('line')['net_incurred_dev'].quantile(CAP_Q)
clm['cost_dev_capped'] = np.minimum(clm['net_incurred_dev'], clm['line'].map(caps_dev))
matched = clm[clm['row_id'].notna()]
agg = matched.groupby('row_id').agg(claim_count=('claim_id', 'size'), cost_reported=('net_incurred', 'sum'),
                                    cost_dev=('net_incurred_dev', 'sum'), cost_dev_capped=('cost_dev_capped', 'sum'))
AGG_COLS = list(agg.columns)

addresses = pd.read_parquet(f'{DATA}/addresses.parquet')
customers = pd.read_parquet(f'{DATA}/customers.parquet').drop(columns=['surname', 'date_of_birth'])
weather = pd.read_parquet(f'{DATA}/weather.parquet').rename(columns={'year': 'policy_year'})
sub_template = pd.read_parquet(f'{DATA}/sample_submission.parquet')[['row_id']]

def std_exposure(df, tol=0.00006):
    """Same rule as home_insurance_model: keep days/365 rows, convert day-count and 360-basis rows to days/365."""
    days = (pd.to_datetime(df['period_end']) - pd.to_datetime(df['period_start'])).dt.days + 1
    exp = df['exposure']
    is_365 = np.isclose(exp, days / 365, atol=tol, rtol=0)
    is_days = ~is_365 & (exp >= 1) & np.isclose(exp, days, atol=0.5, rtol=0)
    is_360 = ~is_365 & ~is_days & np.isclose(exp, days / 360, atol=tol, rtol=0)
    assert (is_365 | is_days | is_360).all(), 'unrecognised exposure convention'
    return np.where(is_365, exp, days / 365)

EXCLUDE = {'row_id', 'policy_id', 'customer_id', 'address_id', 'period_start', 'period_end', 'exposure',
           'exposure_standardised', 'written_premium', 'earned_premium', 'renewal_decision', 'policy_type',
           'policy_year', 'compulsory_excess', *AGG_COLS}

def build_frame(line, kind):
    df = pd.read_parquet(f'{DATA}/{kind}_{line}.parquet')
    df['policy_type'] = line
    df['exposure_standardised'] = std_exposure(df)
    df['earned_premium'] = df['exposure_standardised'] * df['written_premium']
    df['period_start'] = pd.to_datetime(df['period_start']); df['period_end'] = pd.to_datetime(df['period_end'])
    df['policy_year'] = df['period_start'].dt.year
    n0 = len(df)
    df = (df.merge(addresses, on='address_id', how='left', validate='m:1')
            .merge(customers, on='customer_id', how='left', validate='m:1')
            .merge(weather, on=['address_id', 'policy_year'], how='left', validate='m:1'))
    assert len(df) == n0, f'{kind}_{line}: join changed row count'
    df['property_age'] = df['policy_year'] - df['year_built']
    df['customer_tenure'] = df['policy_year'] - df['customer_since_year']
    if 'construction_type' in df:
        df['construction_type'] = df['construction_type'].str.lower().replace({'t': 'code_t', 'w': 'code_w'})
    if kind == 'policies':
        df = df.merge(agg, left_on='row_id', right_index=True, how='left').fillna({c: 0.0 for c in AGG_COLS})
    return df

frames, recons = {}, []
for line in LINES:
    train, test = build_frame(line, 'policies'), build_frame(line, 'test')
    feats = [c for c in train.columns if c not in EXCLUDE]
    for col in feats:
        if train[col].dtype == bool or train[col].dtype == object and set(train[col].dropna().unique()) <= {True, False}:
            train[col] = train[col].astype('float'); test[col] = test[col].astype('float')
        elif train[col].dtype == object or col == 'branch_id':
            cats = sorted(train[col].dropna().astype(str).unique())
            train[col] = pd.Categorical(train[col].astype('string'), categories=cats)
            test[col] = pd.Categorical(test[col].astype('string'), categories=cats)
    frames[line] = {'train': train, 'test': test, 'features': feats}
    c, m = clm[clm['line'] == line], matched[matched['line'] == line]
    recons.append({'line': line, 'claims': len(c), 'matched': len(m), 'unmatched': len(c) - len(m),
                   'frame_reconciles': bool(np.isclose(train['cost_dev'].sum(), m['net_incurred_dev'].sum())),
                   'exposure_years': train['exposure_standardised'].sum(), 'cap_dev_CU': caps_dev[line],
                   'cost_reported': train['cost_reported'].sum(), 'cost_dev': train['cost_dev'].sum(),
                   'dev_uplift_%': (train['cost_dev'].sum() / train['cost_reported'].sum() - 1) * 100,
                   'train_rows': len(train), 'test_rows': len(test), 'n_features': len(feats)})
    del train, test; gc.collect()
display(pd.DataFrame(recons).round(2))

The frames reconcile with the original notebook. Exposure-years are identical to v002 (590,560 / 590,859 / 787,157), and every matched developed claim sits on a training row. Development adds 2.3% to buildings, 1.0% to contents and 7.3% to motor, which matches the chain-ladder IBNR.

## Step 2: Loss cost by policy year, reported vs developed
This is the reason for using the better data. Under the reported target, 2024 (and, for motor, 2023) looked cheaper because those claims were not yet fully reported. That pushed the motor trend negative in v005, so it was floored at 0%. The developed target should take most of that dip out.

In [0]:
rows = []
for line in LINES:
    t = frames[line]['train']
    g = t.groupby('policy_year')[['exposure_standardised', 'claim_count', 'cost_reported', 'cost_dev', 'cost_dev_capped']].sum()
    rows.append(pd.DataFrame({'line': line, 'frequency': g['claim_count'] / g['exposure_standardised'],
                              'loss_cost_reported': g['cost_reported'] / g['exposure_standardised'],
                              'loss_cost_dev': g['cost_dev'] / g['exposure_standardised'],
                              'loss_cost_dev_capped': g['cost_dev_capped'] / g['exposure_standardised']}).reset_index())
display(pd.concat(rows).round(4))

## Step 3: Refit the final GLMs and XGBoost on the developed target
* **GLM.** We use the same spec and the same custom chunked IRLS Tweedie solver (p = 1.5, log link, log exposure offset) as `home_insurance_model` Step 6. The **final factor list is fixed** to the one used for v003, so the GLM structure is the same and only the coefficients are re-estimated. Data-driven quantities (sum-insured clip, vehicle-group credibility map) are recomputed on 2021-23 with the new target.
* **XGBoost.** We use the same parameters, the same base-rate offset, early stopping on 2024 and the same balance correction. It is not logged to MLflow here.
* **Holdout.** D² on 2024 is shown *as is* and *level-adjusted*. Because 2024 is now developed, the as-is figure is less distorted than before.

In [0]:
from scipy import stats
from sklearn.metrics import mean_tweedie_deviance, d2_tweedie_score

inf = np.inf

def split(line):
    t = frames[line]['train']
    return t[t['policy_year'] <= 2023], t[t['policy_year'] == 2024]

def d2_scores(y, pred):
    return d2_tweedie_score(y, pred, power=1.5), d2_tweedie_score(y, pred * y.sum() / pred.sum(), power=1.5)

def band(x, edges, labels):
    return pd.cut(x.astype(float), edges, labels=labels, include_lowest=True).astype(str).replace('nan', 'missing')

def cat(x):
    return x.astype(str).replace({'nan': 'missing', '<NA>': 'missing', 'None': 'missing'})

def flood(d):
    return d['flood_zone'].map({1: '1-2', 2: '1-2', 3: '3', 4: '4', 5: '5'}).fillna('missing')

SI_CLIP = {l: tuple(split(l)[0]['sum_insured'].quantile([0.01, 0.99])) for l in ['buildings', 'contents']}
SI_MED = {l: split(l)[0]['sum_insured'].median() for l in ['buildings', 'contents']}

def vehicle_groups(tr, K=1000):
    g = tr.groupby('make_model', observed=True).agg(exp=('exposure_standardised', 'sum'), cost=(TARGET, 'sum'))
    z = g['exp'] / (g['exp'] + K)
    cred = z * (g['cost'] / g['exp']) / (tr[TARGET].sum() / tr['exposure_standardised'].sum()) + (1 - z)
    order = cred.sort_values().index
    cum = g['exp'].reindex(order).cumsum() / g['exp'].sum()
    return pd.cut(cum, [0, .2, .4, .6, .8, 1.0], labels=['VG1', 'VG2', 'VG3', 'VG4', 'VG5'], include_lowest=True).astype(str), cred
VG_MAP, VG_CRED = vehicle_groups(split('motor')[0])

def log_si(d, line):
    lo, hi = SI_CLIP[line]
    return np.log(d['sum_insured'].clip(lo, hi) / SI_MED[line])

GLM_SPEC = {
    'buildings': lambda d: pd.DataFrame({
        'flood_zone': flood(d),
        'elevation': band(d['elevation_m'], [-inf, 1.8, 3.1, 4.6, inf], ['<=1.8m', '1.8-3.1m', '3.1-4.6m', '>4.6m']),
        'river_distance': band(d['distance_to_river_km'], [-inf, 0.45, 0.85, 1.3, inf], ['<=0.45km', '0.45-0.85km', '0.85-1.3km', '>1.3km']),
        'log_sum_insured': log_si(d, 'buildings'),
        'property_age': band(d['property_age'], [-inf, 60, 85, inf], ['<60', '60-85', '85+']),
        'dry_summer': band(d['dry_summer_index'], [-inf, 0.65, 0.8, inf], ['<=0.65', '0.65-0.80', '>0.80']),
        'prior_claims': cat(d['prior_claims_5y'].clip(upper=2).map({0: '0', 1: '1', 2: '2+'})),
        'region': cat(d['region']), 'soil_type': cat(d['soil_type']), 'energy_rating': cat(d['energy_rating']),
        'construction_type': cat(d['construction_type']), 'credit_band': cat(d['credit_band']),
        'voluntary_excess': cat(d['voluntary_excess']), 'channel': cat(d['channel']),
        'flood45_x_low_elev': ((d['flood_zone'] >= 4) & (d['elevation_m'] <= 3.1)).astype(float)}),
    'contents': lambda d: pd.DataFrame({
        'log_sum_insured': log_si(d, 'contents'),
        'flood_zone': flood(d),
        'elevation': band(d['elevation_m'], [-inf, 1.8, 3.1, inf], ['<=1.8m', '1.8-3.1m', '>3.1m']),
        'river_distance': band(d['distance_to_river_km'], [-inf, 0.8, inf], ['<=0.8km', '>0.8km']),
        'voluntary_excess': cat(d['voluntary_excess']),
        'prior_claims': cat(d['prior_claims_5y'].clip(upper=2).map({0: '0', 1: '1', 2: '2+'})),
        'accidental_damage': cat(d['accidental_damage_cover']),
        'high_value_items': cat(d['high_value_items'].clip(upper=3).map({0: '0', 1: '1', 2: '2', 3: '3+'})),
        'tenure': cat(d['tenure']), 'payment_method': cat(d['payment_method']),
        'region': cat(d['region']), 'urban_density': cat(d['urban_density']),
        'property_type': cat(d['property_type']), 'occupation': cat(d['occupation']),
        'flood45_x_low_elev': ((d['flood_zone'] >= 4) & (d['elevation_m'] <= 3.1)).astype(float)}),
    'motor': lambda d: pd.DataFrame({
        'ncb_years': band(d['ncb_years'], [-inf, 1, 3, 5, 7, inf], ['0-1', '2-3', '4-5', '6-7', '8+']),
        'cover_type': cat(d['cover_type']),
        'annual_mileage': band(d['annual_mileage'], [-inf, 5000, 10000, 15000, 20000, inf], ['<=5k', '5-10k', '10-15k', '15-20k', '>20k']),
        'driver_age': band(d['driver_age'], [-inf, 22, 28, 35, 72, 80, inf], ['17-22', '23-28', '29-35', '36-72', '73-80', '81+']),
        'vehicle_group': d['make_model'].astype(str).map(VG_MAP).fillna('VG3'),
        'years_licensed': band(d['years_licensed'], [-inf, 2, inf], ['0-2', '3+']),
        'vehicle_age': band(d['vehicle_age'], [-inf, 7, inf], ['<=7', '8+']),
        'telematics': np.where(d['telematics_score'].isna(), 'not_opted_in',
                               band(d['telematics_score'], [-inf, 50, 65, inf], ['<50', '50-65', '65+'])),
        'prior_claims': cat(d['prior_claims_5y'].clip(upper=2).map({0: '0', 1: '1', 2: '2+'})),
        'voluntary_excess': cat(d['voluntary_excess']), 'region': cat(d['region']), 'channel': cat(d['channel']),
        'payment_method': cat(d['payment_method']), 'urban_density': cat(d['urban_density']),
        'vehicle_use': cat(d['vehicle_use']),
        'missed_payments': cat(d['missed_payments_12m'].clip(upper=2).map({0: '0', 1: '1', 2: '2+'})),
        'licence_points': d['licence_points'].fillna(0).clip(upper=6).map({0: '0', 3: '3', 6: '6+'})}),
}
# Final factor list exactly as used for home_insurance_submission_v003 (non-significant factors at 1% dropped)
GLM_DROP = {'buildings': ['river_distance', 'dry_summer', 'flood45_x_low_elev'],
            'contents': ['river_distance', 'tenure', 'property_type', 'occupation', 'flood45_x_low_elev'],
            'motor': []}

def design(F, exposure, levels=None):
    """One-hot encode with the largest-exposure level as base; reuse training levels for other data."""
    if levels is None:
        levels = {c: (F.groupby(c)[c].size().index.tolist(),
                      pd.Series(exposure.values, index=F.index).groupby(F[c]).sum().idxmax())
                  for c in F if F[c].dtype == object}
    cols = {'const': np.ones(len(F), dtype=np.float32)}
    for c in F:
        if c in levels:
            lv, base = levels[c]
            for l in lv:
                if l != base:
                    cols[f'{c}[{l}]'] = (F[c] == l).values.astype(np.float32)
        else:
            cols[c] = F[c].values.astype(np.float32)
    return pd.DataFrame(cols, index=F.index), levels

class TweedieGLM:
    """Memory-light IRLS Tweedie GLM, log link, offset (as in home_insurance_model, verified there vs statsmodels)."""
    def __init__(self, p=1.5, tol=1e-9, max_iter=50, chunk=100_000):
        self.p, self.tol, self.max_iter, self.chunk = p, tol, max_iter, chunk

    def _xtwx(self, X, w, z=None):
        k = X.shape[1]; A = np.zeros((k, k)); b = np.zeros(k)
        for i in range(0, len(X), self.chunk):
            Xi = X[i:i + self.chunk].astype(np.float64); wi = w[i:i + self.chunk]
            A += (Xi * wi[:, None]).T @ Xi
            if z is not None:
                b += Xi.T @ (wi * z[i:i + self.chunk])
        return A, b

    def fit(self, X, y, offset):
        Xv, y, off, p = X.to_numpy(np.float32), np.asarray(y, float), np.asarray(offset, float), self.p
        beta = np.zeros(Xv.shape[1]); beta[0] = np.log(y.sum() / np.exp(off).sum())
        dev_old = np.inf
        for it in range(self.max_iter):
            eta = Xv @ beta.astype(np.float32) + off
            mu = np.exp(eta)
            w = mu ** (2 - p)
            A, b = self._xtwx(Xv, w, eta - off + (y - mu) / mu)
            beta = np.linalg.solve(A, b)
            mu = np.exp(Xv @ beta.astype(np.float32) + off)
            dev = mean_tweedie_deviance(y, mu, power=p) * len(y)
            if abs(dev_old - dev) / dev < self.tol:
                break
            dev_old = dev
        self.n_iter, self.nobs = it + 1, len(y)
        self.scale = np.sum((y - mu) ** 2 / mu ** p) / (len(y) - len(beta))
        A, _ = self._xtwx(Xv, mu ** (2 - p))
        cov = self.scale * np.linalg.inv(A)
        self.params = pd.Series(beta, index=X.columns)
        self.cov = pd.DataFrame(cov, index=X.columns, columns=X.columns)
        return self

    def predict(self, X, offset):
        return np.exp(X.to_numpy(np.float32) @ self.params.values.astype(np.float32) + np.asarray(offset, float))

def glm_predict(g, line, df):
    X, _ = design(GLM_SPEC[line](df).drop(columns=g['drop']), df['exposure_standardised'], levels=g['levels'])
    return g['res'].predict(X, np.log(df['exposure_standardised']))

glm_final, glm_tests = {}, []
for line in LINES:
    t0 = time.time()
    tr, _ = split(line)
    F = GLM_SPEC[line](tr).drop(columns=GLM_DROP[line])
    X, levels = design(F, tr['exposure_standardised'])
    res = TweedieGLM().fit(X, tr[TARGET], np.log(tr['exposure_standardised']))
    glm_final[line] = {'res': res, 'levels': levels, 'drop': GLM_DROP[line]}
    for f in F:
        idx = [c for c in X.columns if c == f or c.startswith(f + '[')]
        b = res.params[idx].values; V = res.cov.loc[idx, idx].values
        chi2 = float(b @ np.linalg.solve(V, b))
        glm_tests.append({'line': line, 'factor': f, 'df': len(idx), 'wald_chi2': chi2, 'p_value': stats.chi2.sf(chi2, len(idx))})
    print(f"{line}: {X.shape[1]} parameters, {res.n_iter} IRLS iterations, base rate {np.exp(res.params['const']):,.1f} CU/yr, {time.time() - t0:.0f}s")
    del X, F; gc.collect()
glm_tests = pd.DataFrame(glm_tests)
not_sig = glm_tests[glm_tests['p_value'] >= 0.01]
print(f'{len(glm_tests) - len(not_sig)} of {len(glm_tests)} final factors still significant at 1% on the developed target')
if len(not_sig):
    display(not_sig.round(4))

In [0]:
import xgboost as xgb

LEAKAGE = {'customer_service_contacts'}
FEATS = {l: [f for f in frames[l]['features'] if f not in LEAKAGE] for l in LINES}
XGB_PARAMS = {'objective': 'reg:tweedie', 'tweedie_variance_power': 1.5, 'eval_metric': 'tweedie-nloglik@1.5',
              'eta': 0.05, 'max_depth': 5, 'min_child_weight': 1000, 'subsample': 0.8, 'colsample_bytree': 0.8,
              'lambda': 1.0, 'tree_method': 'hist', 'max_cat_to_onehot': 1, 'nthread': 4, 'seed': 42}
BASE_RATE = {l: split(l)[0][TARGET].sum() / split(l)[0]['exposure_standardised'].sum() for l in LINES}

def dmatrix(df, line, with_label=True):
    return xgb.DMatrix(df[FEATS[line]], label=df[TARGET] if with_label else None,
                       base_margin=np.log(df['exposure_standardised'] * BASE_RATE[line]), enable_categorical=True)

xgb_models, lift_data, comparison = {}, {}, []
for line in LINES:
    t0 = time.time()
    tr, ho = split(line)
    dtr, dho = dmatrix(tr, line), dmatrix(ho, line)
    bst = xgb.train(XGB_PARAMS, dtr, num_boost_round=1500, evals=[(dho, 'holdout_2024')],
                    early_stopping_rounds=50, verbose_eval=False)
    it = (0, bst.best_iteration + 1)
    cal = tr[TARGET].sum() / bst.predict(dtr, iteration_range=it).sum()       # balance correction, as before
    xgb_models[line] = {'booster': bst, 'iteration_range': it, 'balance_factor': cal}
    h = ho[['row_id', 'exposure_standardised', TARGET]].copy()
    h['pred_xgb'] = bst.predict(dho, iteration_range=it) * cal
    h['pred_glm'] = glm_predict(glm_final[line], line, ho)
    lift_data[line] = h
    for name, col in [('GLM final', 'pred_glm'), ('XGBoost', 'pred_xgb')]:
        d2, d2a = d2_scores(h[TARGET], h[col])
        comparison.append({'line': line, 'model': name, 'trees': it[1] if col == 'pred_xgb' else None,
                           'D2_2024': d2, 'D2_2024_level_adj': d2a, 'pred/actual 2024': h[col].sum() / h[TARGET].sum()})
    print(f'{line}: XGBoost {it[1]} trees, balance factor {cal:.3f}, {(time.time() - t0) / 60:.1f} min')
    del dtr, dho; gc.collect()
comparison = pd.DataFrame(comparison)
display(comparison.round(4))

## Step 4: `chainladder_submission_v001` (the v003 approach)
We apply the final GLM (refitted on the developed target) to the 2025 test rows. Each line is then rescaled so that its total equals the v002 total (v002 rate × 2025 test exposure: 420.28 / 111.92 / 548.46 CU per year), exactly as for v003. Because the level is the same, any score difference from v003 comes **only** from the developed target changing the relativities. The file checks are the same as before: 626,639 rows in `sample_submission` order, no nulls, every prediction > 0 (floored at 0.01), saved as CSV only.

In [0]:
V002_RATE = {'buildings': 420.28, 'contents': 111.92, 'motor': 548.46}  # CU per policy year (v002)

def save_submission(parts, name):
    sub = sub_template.merge(pd.concat(parts), on='row_id', how='left', validate='one_to_one')
    sub['prediction'] = sub['prediction'].clip(lower=0.01).astype('float64').round(2)
    assert len(sub) == 626_639 and sub['row_id'].is_unique and (sub['row_id'].values == sub_template['row_id'].values).all()
    assert sub['prediction'].notna().all() and (sub['prediction'] > 0).all()
    path = f'{OUT_DIR}/{name}.csv'
    sub.to_csv(path, index=False, float_format='%.2f')
    chk = pd.read_csv(path, dtype={'row_id': str})        # re-read what Kaggle will receive
    assert chk['row_id'].equals(sub_template['row_id']) and chk['prediction'].notna().all() and (chk['prediction'] > 0).all()
    print(f'Saved {len(chk):,} rows to {os.path.abspath(path)} | min {chk.prediction.min():.2f}, mean {chk.prediction.mean():.2f}')
    return chk

def compare_with(new, old_file, label):
    old = pd.read_csv(f'{OUT_DIR}/{old_file}', dtype={'row_id': str})
    assert old['row_id'].equals(new['row_id'])
    d = pd.DataFrame({'line': new['row_id'].str[0].map(policy_type_map), 'new': new['prediction'], 'old': old['prediction']})
    return d.groupby('line').apply(lambda g: pd.Series({f'{label} total': g['old'].sum(), 'new total': g['new'].sum(),
                                                       'new / old': g['new'].sum() / g['old'].sum(),
                                                       'correlation': g['new'].corr(g['old'])}), include_groups=False).reset_index()

parts, info = [], []
for line in LINES:
    te = frames[line]['test']
    e = te['exposure_standardised'].to_numpy()
    p = np.asarray(glm_predict(glm_final[line], line, te), dtype=float)
    assert np.isfinite(p).all() and (p > 0).all()
    scale = V002_RATE[line] * e.sum() / p.sum()
    parts.append(pd.DataFrame({'row_id': te['row_id'].values, 'prediction': p * scale}))
    rate = p * scale / e
    info.append({'line': line, 'scale_to_v002_level': scale, 'total_CU': p.sum() * scale,
                 'rate_p5': np.percentile(rate, 5), 'rate_median': np.median(rate), 'rate_p95': np.percentile(rate, 95)})
display(pd.DataFrame(info).round(3))
cl_v001 = save_submission(parts, 'chainladder_submission_v001')
display(compare_with(cl_v001, 'home_insurance_submission_v003.csv', 'v003').round(4))

## Step 5: `chainladder_submission_v002` (the v005 approach)
The method is the same as v005, but every input comes from the developed target:
1. **Blend** GLM and XGBoost. The GLM weight is chosen on the 2024 holdout (level-adjusted D², grid 0 to 1 in steps of 0.1).
2. **Large-loss loading** = developed uncapped cost / developed capped cost, over 2021-23.
3. **Trend to 2025** = (1 + g)^3. Here g is the log-linear annual change in developed capped loss cost over 2021-23, limited to the range 0% to +8%.

Now that 2021-23 is developed, the level should no longer be short by the IBNR that v005 was missing. We then compare the result with v005.

In [0]:
W_GRID = np.round(np.arange(0, 1.01, 0.1), 1)
parts, info = [], []
for line in LINES:
    h = lift_data[line]; y = h[TARGET]
    scores = {w: d2_scores(y, w * h['pred_glm'] + (1 - w) * h['pred_xgb'])[1] for w in W_GRID}
    w = max(scores, key=scores.get)

    tr, _ = split(line)
    ll = tr['cost_dev'].sum() / tr[TARGET].sum()
    yr = tr.groupby('policy_year').agg(c=(TARGET, 'sum'), e=('exposure_standardised', 'sum'))
    lc = yr['c'] / yr['e']
    g_raw = np.exp(np.polyfit(lc.index.values, np.log(lc.values), 1)[0]) - 1
    g = float(np.clip(g_raw, 0.0, 0.08))
    trend = (1 + g) ** (2025 - 2022)

    te = frames[line]['test']
    p_glm = np.asarray(glm_predict(glm_final[line], line, te), dtype=float)
    xm = xgb_models[line]
    p_xgb = xm['booster'].predict(dmatrix(te, line, with_label=False), iteration_range=xm['iteration_range']) * xm['balance_factor']
    p = (w * p_glm + (1 - w) * p_xgb) * ll * trend
    assert np.isfinite(p).all() and (p > 0).all()
    parts.append(pd.DataFrame({'row_id': te['row_id'].values, 'prediction': p}))
    info.append({'line': line, 'GLM_weight': w, 'D2_GLM_2024': scores[1.0], 'D2_XGB_2024': scores[0.0], 'D2_blend_2024': scores[w],
                 'loss_cost_2021': lc.iloc[0], 'loss_cost_2023': lc.iloc[-1], 'trend_g_raw': g_raw, 'trend_g_used': g,
                 'large_loss_loading': ll, 'trend_factor': trend, 'rate_CU': p.sum() / te['exposure_standardised'].sum(),
                 'v005_rate_CU': {'buildings': 522.07, 'contents': 130.36, 'motor': 532.24}[line], 'v002_rate_CU': V002_RATE[line]})
display(pd.DataFrame(info).round(4))
cl_v002 = save_submission(parts, 'chainladder_submission_v002')
display(compare_with(cl_v002, 'home_insurance_submission_v005.csv', 'v005').round(4))

## Findings
* **`chainladder_submission_v001` is almost the same as v003.** The correlation per line is 1.000 / 1.000 / 0.9999, and the totals are the same by design. The models are trained on 2021-23, and for those years the Benktander factors are 1.000-1.004 for property and 1.003-1.032 for motor, so the developed target hardly moves the relativities. All 39 final GLM factors are still significant at 1%. Expect a leaderboard score very close to v003's.
* **`chainladder_submission_v002` is v005 plus a small level increase.** The rates are 526.8 / 130.8 / 541.1 CU per year, against 522.1 / 130.4 / 532.2 for v005. That is +0.9% buildings, +0.4% contents and +1.7% motor, with correlation 0.997-0.999. This is the IBNR left in 2021-23. Blend weights are 0.8 / 0.7 / 0.8: buildings moves from 0.7 to 0.8, the others are unchanged.
* **The motor trend is still floored at 0%.** Developed *uncapped* motor loss cost rises (565 → 588 → 606 CU per year over 2021-23). The *capped* version used for the trend falls in 2023 (546 → 513), so 2023 has more of its cost above the cap. The raw trend is -1.2% a year (it was -2.5%), so it is floored at 0%.
* **The developed data matters most for 2024.** Motor 2024 loss cost goes from 454 (reported) to 587 CU per year (developed), so 2024 is now usable for training. The bigger gain is probably to refit on **2021-24 developed** data, which adds a fourth, more recent year, and to estimate the trend on uncapped or 2021-24 data. That would be a separate version.